# HMC Crown vigilance compare (Head A-vig, frozen CBraMod)

**Public retrain path:** Hugging Face [`windwerfer/neurofeed-eeg-windows`](https://huggingface.co/datasets/windwerfer/neurofeed-eeg-windows)
configs `crown2_vigilance_hmc` / `crown4_vigilance_hmc` (CC-BY-4.0) + subject splits in this repo under
`datasets/vigilance_hmc_crown2|4/splits/`. Scripts: `scripts/train_hmc_crown_vig_compare.py`.

Montages **crown2_strong** (C=2: C3,C4) vs **crown4_hmc** (C=4: C3,C4,F6,PO4; honest proxy F6≈F4, PO4≈O2).
True channel count (no zero-pad to 8). Ship bar: test macro-F1 ≥ 0.60.

**Maintainer note:** Private Kaggle datasets/kernels are optional GPU scratch only — not required for public use.
Do **not** publish ISRUC, L-FAME, LUNA, or SEED-VIG here.


In [ ]:
import os, sys, json, shutil
from pathlib import Path

# Prefer a local checkout of neurofeed_train; fall back to Kaggle working dir.
CANDIDATE_ROOTS = [
    Path('/kaggle/working/neurofeed_train'),
    Path('.').resolve(),
    Path('/kaggle/working'),
]
ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / 'scripts' / 'train_hmc_crown_vig_compare.py').exists():
        ROOT = cand
        break
if ROOT is None:
    ROOT = Path('/kaggle/working/neurofeed_train')
    ROOT.mkdir(parents=True, exist_ok=True)
    print('WARN: train script not found yet; clone/copy neurofeed_train into', ROOT)

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
INPUT = Path('/kaggle/input')
print('ROOT', ROOT)
print('inputs', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else None)


In [ ]:
from pathlib import Path
import os, shutil

def first(*cands):
    for c in cands:
        if c is None:
            continue
        p = Path(c)
        if p.exists():
            return p
    return None

# Public HF mirror (preferred). Map HF config names -> local pack dirs expected by the train script.
HF_ROOT = first(
    os.environ.get('HF_WINDOWS_ROOT'),
    os.environ.get('DATASETS_ROOT'),
    ROOT / 'data' / 'hf_windows',
    Path('./data/hf_windows'),
)
HF_MAP = {
    'vigilance_hmc_crown2': 'crown2_vigilance_hmc',
    'vigilance_hmc_crown4': 'crown4_vigilance_hmc',
}

# Optional private Kaggle scratch (maintainers only) — never required for public retrain.
CROWN = first('/kaggle/input/muse-eeg-heads-crown-vig')
CACHE = first('/kaggle/input/muse-eeg-heads-cache')

(ROOT / 'datasets').mkdir(parents=True, exist_ok=True)
for pack, hf_cfg in HF_MAP.items():
    dest = ROOT / 'datasets' / pack
    dest.mkdir(parents=True, exist_ok=True)
    win_dest = dest / 'windows'
    # Prefer HF mirror windows/
    src = None
    if HF_ROOT:
        for cand in [HF_ROOT / hf_cfg / 'windows', HF_ROOT / hf_cfg]:
            if cand.exists():
                src = cand if cand.name == 'windows' else (cand / 'windows' if (cand / 'windows').exists() else cand)
                break
    if src is None and CROWN:
        for cand in [CROWN / 'datasets' / pack / 'windows', CROWN / 'datasets' / pack]:
            if cand.exists():
                src = cand if cand.name == 'windows' else (cand / 'windows' if (cand / 'windows').exists() else cand)
                break
    if src is None:
        print(f'MISSING windows for {pack} ({hf_cfg}). Download HF config and set HF_WINDOWS_ROOT.')
        continue
    if win_dest.exists() and any(win_dest.glob('*_windows.npz')):
        n = len(list(win_dest.glob('*_windows.npz')))
        print(pack, 'already staged nights', n)
        continue
    if win_dest.exists():
        shutil.rmtree(win_dest)
    if src.name == 'windows':
        shutil.copytree(src, win_dest)
    else:
        win_dest.mkdir(parents=True, exist_ok=True)
        for f in src.glob('*_windows.npz'):
            shutil.copy2(f, win_dest / f.name)
    n = len(list(win_dest.glob('*_windows.npz')))
    print(pack, 'nights', n, 'from', src)

# CBraMod weights: env, local data/, or optional private Kaggle cache (not redistributed).
w = first(
    os.environ.get('CBRAMOD_WEIGHTS'),
    ROOT / 'data' / 'models' / 'CBraMod' / 'pretrained_weights.pth',
    CACHE / 'models' / 'CBraMod' / 'pretrained_weights.pth' if CACHE else None,
    CROWN / 'models' / 'CBraMod' / 'pretrained_weights.pth' if CROWN else None,
)
assert w and Path(w).exists(), (
    'CBraMod pretrained_weights.pth missing. Set CBRAMOD_WEIGHTS or place under data/models/CBraMod/.'
)
print('weights', w, Path(w).stat().st_size)
os.environ.setdefault('CBRAMOD_WEIGHTS', str(w))


In [ ]:
import torch, sys
sys.path.insert(0, str(ROOT))
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
from src.cbramod_encoder import FrozenCBraModEncoder
enc = FrozenCBraModEncoder(Path(os.environ['CBRAMOD_WEIGHTS']), source_sr=256.0, pool='mean', map_location='cpu')
for c in (2, 4):
    y = enc(torch.zeros(1, c, 512))
    print(f'smoke C={c} -> {tuple(y.shape)}')
del enc


In [ ]:
import runpy, os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
out = ROOT / 'exports' / 'hmc_crown_vig_compare'
sys.argv = ['train_hmc_crown_vig_compare.py', '--out', str(out)]
runpy.run_path(str(ROOT / 'scripts' / 'train_hmc_crown_vig_compare.py'), run_name='__main__')


In [ ]:
import json, shutil
from pathlib import Path
cmp_path = ROOT / 'exports' / 'hmc_crown_vig_compare' / 'compare.json'
cmp = json.loads(cmp_path.read_text())
summary = {
    'any_ship_candidate': cmp.get('any_ship_candidate'),
    'both_below_bar': cmp.get('both_below_bar'),
    'f1': {
        r['montage']: {
            'val_macro_f1': r['metrics']['val']['macro_f1'],
            'test_macro_f1': r['metrics']['test']['macro_f1'],
            'ship_candidate': r['ship_candidate'],
            'n_channels': r['n_channels'],
        }
        for r in cmp['results']
    },
}
print(json.dumps(summary, indent=2))
(WORKING / 'compare_summary.json').write_text(json.dumps(summary, indent=2))
shutil.copy2(cmp_path, WORKING / 'compare.json')
